# Operational Network Preparation

Build and review the fixed-asset transmission topology for the mu-star energy model.

This notebook stops at an explicit readiness gate. It will not run dispatch until line ratings, existing generator capacities, bus assignments and calibrated demand are complete.

In [ ]:
from pathlib import Path
import json

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

from mu_star_energy.distribution import build_service_weights
from mu_star_energy.paths import incoming_energy_dir, processed_energy_dir
from mu_star_energy.topology import build_substation_topology

COLLABORATOR_DIR = processed_energy_dir() / "collaborator"
NETWORK_DIR = processed_energy_dir() / "network"
NETWORK_DIR.mkdir(parents=True, exist_ok=True)

substations = gpd.read_parquet(COLLABORATOR_DIR / "substations.parquet")
routes = gpd.read_parquet(COLLABORATOR_DIR / "transmission_routes.parquet")
result = build_substation_topology(substations, routes, snap_tolerance_m=2500, default_voltage_kv=66)

result.buses.to_parquet(NETWORK_DIR / "buses.parquet")
result.lines.to_parquet(NETWORK_DIR / "lines.parquet")
report = {
    "buses": len(result.buses),
    "lines": len(result.lines),
    "ignored_route_parts": result.ignored_route_parts,
    "line_ratings_complete": bool(not result.lines.empty and result.lines["s_nom_mva"].notna().all()),
}
(NETWORK_DIR / "topology_report.json").write_text(json.dumps(report, indent=2))
report

## Inferred Transmission Topology

Routes are exploded into line parts. Substations near each route are projected onto it and consecutive substations are connected. This creates a reproducible provisional graph, not a validated CEB bus-branch model.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
routes.plot(ax=ax, color="#d1d5db", linewidth=3, alpha=0.55, label="source route geometry")
if not result.lines.empty:
    result.lines.plot(ax=ax, color="#dc2626", linewidth=1.6, label="inferred branch")
result.buses.plot(ax=ax, color="#ff8c00", edgecolor="black", markersize=50, label="substation")
for _, row in result.buses.iterrows():
    ax.annotate(row["bus_id"], (row.geometry.x, row.geometry.y), xytext=(3, 3), textcoords="offset points", fontsize=7)
ax.set_title("Provisional substation topology")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(alpha=0.2)
ax.legend()
plt.show()

display(result.lines.drop(columns="geometry"))

## Distribution Service Proxy

OSM and GridFinder lines are optional. When present, their line lengths are assigned to the nearest transmission substation and converted to service weights. This is for demand/customer allocation only; inferred lines are not inserted into the PyPSA electrical network.

In [ ]:
def optional_line_layer(path):
    if not path.exists():
        return None
    return gpd.read_parquet(path) if path.suffix == ".parquet" else gpd.read_file(path)

gridfinder_path = incoming_energy_dir() / "gridfinder" / "grid.gpkg"
osm_path = incoming_energy_dir() / "osm" / "distribution_lines.parquet"
service_weights = build_service_weights(
    result.buses,
    gridfinder_lines=optional_line_layer(gridfinder_path),
    osm_distribution_lines=optional_line_layer(osm_path),
)
service_weights.to_csv(COLLABORATOR_DIR / "service_weights.csv", index=False)
display(service_weights)

## Operational Readiness Gate

The model deliberately refuses to optimise missing infrastructure. Populate these fields before building the PyPSA network.

In [ ]:
generator_register = pd.read_csv(COLLABORATOR_DIR / "generation_register_template.csv")
readiness = pd.Series({
    "line ratings populated": int(result.lines["s_nom_mva"].notna().sum()),
    "line ratings required": len(result.lines),
    "generator capacities populated": int(generator_register["capacity_mw"].notna().sum()),
    "generator capacities required": len(generator_register),
    "generator bus assignments populated": int(generator_register["connected_bus_id"].notna().sum()),
    "generator bus assignments required": len(generator_register),
    "calibrated demand profile present": int((COLLABORATOR_DIR / "demand_profile.csv").exists()),
    "distribution proxy source": service_weights["method"].iloc[0] if len(service_weights) else "none",
})
display(readiness.to_frame("value"))

## Disruption Input Contract

After the readiness gate is complete, mu-star supplies a table such as:

| component | asset_id | available_fraction |
|---|---|---:|
| Line | LINE_004 | 0.0 |
| Generator | Fort_George_1 | 0.4 |
| Bus | SUB_008 | 0.0 |

`EnergyModel.simulate(network, disruptions)` copies the fixed network, applies availability reductions, redispatches generation, and returns unserved energy, served fraction and operating cost. Damage curves are upstream of this interface.